In [1]:
import sys
!{sys.executable} -m pip install reportlab

In [2]:
import reportlab
print("ReportLab is working!")

ReportLab is working!


In [1]:
import os
import csv
import shutil
import sqlite3
import tkinter as tk
from tkinter import ttk, messagebox, filedialog
from datetime import datetime, date
from email.message import EmailMessage
import smtplib
from io import BytesIO
try:
    from reportlab.lib import colors
    from reportlab.lib.pagesizes import A4
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.enums import TA_CENTER
    from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, PageBreak
    REPORTLAB_AVAILABLE = True
except ImportError:
    REPORTLAB_AVAILABLE = False


In [2]:
DB_FILE = "doctor_management.db"
APP_TITLE = "Doctor Management System"
DATE_FORMAT = "%Y-%m-%d"
TIME_FORMAT = "%H:%M"
SMTP_SERVER = "smtp.gmail.com"
SMTP_PORT = 465
SMTP_USERNAME = ""       # Example: yourname@gmail.com
SMTP_PASSWORD = ""       # Gmail App Password


In [3]:
class Database:
    """SQLite database layer. Keeps the original Version-1 data and adds Version-2 fields."""

    def __init__(self, db_file=DB_FILE):
        self.db_file = db_file
        self.connection = sqlite3.connect(db_file)
        self.connection.row_factory = sqlite3.Row
        self.connection.execute("PRAGMA foreign_keys = ON")
        self.create_tables()
        self.seed_admin()

    def _columns(self, table):
        return {row[1] for row in self.connection.execute(f"PRAGMA table_info({table})")}

    def _add_column(self, table, column, definition):
        if column not in self._columns(table):
            self.connection.execute(f"ALTER TABLE {table} ADD COLUMN {column} {definition}")

    def create_tables(self):
        self.connection.executescript("""
            CREATE TABLE IF NOT EXISTS users (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                username TEXT UNIQUE NOT NULL,
                password TEXT NOT NULL,
                role TEXT NOT NULL DEFAULT 'Admin',
                created_at TEXT DEFAULT CURRENT_TIMESTAMP
            );

            CREATE TABLE IF NOT EXISTS patients (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                name TEXT NOT NULL,
                age INTEGER,
                gender TEXT,
                phone TEXT,
                address TEXT,
                email TEXT,
                blood_group TEXT,
                allergies TEXT,
                emergency_contact TEXT,
                created_at TEXT DEFAULT CURRENT_TIMESTAMP
            );

            CREATE TABLE IF NOT EXISTS doctors (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                name TEXT NOT NULL,
                specialization TEXT,
                phone TEXT,
                room TEXT,
                email TEXT,
                availability TEXT,
                created_at TEXT DEFAULT CURRENT_TIMESTAMP
            );

            CREATE TABLE IF NOT EXISTS appointments (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                patient_id INTEGER NOT NULL,
                doctor_id INTEGER NOT NULL,
                appointment_date TEXT NOT NULL,
                appointment_time TEXT NOT NULL,
                reason TEXT,
                status TEXT DEFAULT 'Scheduled',
                notes TEXT,
                created_at TEXT DEFAULT CURRENT_TIMESTAMP,
                FOREIGN KEY(patient_id) REFERENCES patients(id) ON DELETE CASCADE,
                FOREIGN KEY(doctor_id) REFERENCES doctors(id) ON DELETE CASCADE
            );

            CREATE TABLE IF NOT EXISTS consultations (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                appointment_id INTEGER UNIQUE NOT NULL,
                symptoms TEXT,
                diagnosis TEXT,
                prescription TEXT,
                doctor_notes TEXT,
                blood_pressure TEXT,
                temperature TEXT,
                weight TEXT,
                notes TEXT,
                follow_up_date TEXT,
                created_at TEXT DEFAULT CURRENT_TIMESTAMP,
                FOREIGN KEY(appointment_id) REFERENCES appointments(id) ON DELETE CASCADE
            );

            CREATE TABLE IF NOT EXISTS prescriptions (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                consultation_id INTEGER NOT NULL,
                medicine TEXT NOT NULL,
                dosage TEXT,
                frequency TEXT,
                duration TEXT,
                instructions TEXT,
                FOREIGN KEY(consultation_id) REFERENCES consultations(id) ON DELETE CASCADE
            );

            CREATE INDEX IF NOT EXISTS idx_patient_name ON patients(name);
            CREATE INDEX IF NOT EXISTS idx_patient_phone ON patients(phone);
            CREATE INDEX IF NOT EXISTS idx_doctor_name ON doctors(name);
            CREATE INDEX IF NOT EXISTS idx_appointment_date ON appointments(appointment_date);
        """)

        # Migrate databases created by the original application.
        for table, additions in {
            "patients": [
                ("email", "TEXT"), ("blood_group", "TEXT"),
                ("allergies", "TEXT"), ("emergency_contact", "TEXT")
            ],
            "doctors": [("email", "TEXT"), ("availability", "TEXT")],
            "appointments": [("notes", "TEXT")],
            "consultations": [
                ("blood_pressure", "TEXT"), ("temperature", "TEXT"),
                ("weight", "TEXT"), ("notes", "TEXT")
            ]
        }.items():
            for column, definition in additions:
                self._add_column(table, column, definition)

        # Preserve old doctor_notes data when the new notes field is introduced.
        if "doctor_notes" in self._columns("consultations"):
            self.connection.execute("""
                UPDATE consultations
                SET notes = doctor_notes
                WHERE (notes IS NULL OR notes = '')
                  AND doctor_notes IS NOT NULL
            """)

        self.connection.commit()

    def seed_admin(self):
        self.connection.execute("""
            INSERT OR IGNORE INTO users(username, password, role)
            VALUES ('admin', 'admin123', 'Admin')
        """)
        self.connection.commit()

    def close(self):
        if self.connection:
            self.connection.close()

    # ---------------- Authentication ----------------
    def login(self, username, password):
        return self.connection.execute("""
            SELECT * FROM users WHERE username = ? AND password = ?
        """, (username, password)).fetchone()

    def change_password(self, user_id, current_password, new_password):
        user = self.connection.execute(
            "SELECT password FROM users WHERE id=?", (user_id,)
        ).fetchone()
        if not user or user["password"] != current_password:
            return False
        self.connection.execute(
            "UPDATE users SET password=? WHERE id=?",
            (new_password, user_id)
        )
        self.connection.commit()
        return True

    def change_username(self, user_id, current_password, new_username):
        """Change the username for the currently logged-in user."""
        user = self.connection.execute(
            "SELECT password FROM users WHERE id=?", (user_id,)
        ).fetchone()
        if not user or user["password"] != current_password:
            return False, "Current password is incorrect."

        new_username = new_username.strip()
        if not new_username:
            return False, "Username cannot be empty."
        if len(new_username) < 3:
            return False, "Username must contain at least 3 characters."

        existing = self.connection.execute(
            "SELECT id FROM users WHERE username=? AND id<>?",
            (new_username, user_id)
        ).fetchone()
        if existing:
            return False, "That username is already in use."

        self.connection.execute(
            "UPDATE users SET username=? WHERE id=?",
            (new_username, user_id)
        )
        self.connection.commit()
        return True, new_username

    # ---------------- Patients ----------------
    def add_patient(self, data):
        self.connection.execute("""
            INSERT INTO patients
            (name, age, gender, phone, address, email, blood_group, allergies, emergency_contact)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, data)
        self.connection.commit()

    def update_patient(self, patient_id, data):
        self.connection.execute("""
            UPDATE patients SET
                name=?, age=?, gender=?, phone=?, address=?, email=?,
                blood_group=?, allergies=?, emergency_contact=?
            WHERE id=?
        """, (*data, patient_id))
        self.connection.commit()

    def delete_patient(self, patient_id):
        self.connection.execute("DELETE FROM patients WHERE id=?", (patient_id,))
        self.connection.commit()

    def get_patients(self, search=""):
        if search:
            term = f"%{search}%"
            return self.connection.execute("""
                SELECT * FROM patients
                WHERE name LIKE ? OR phone LIKE ? OR email LIKE ? OR CAST(id AS TEXT) LIKE ?
                ORDER BY name
            """, (term, term, term, term)).fetchall()
        return self.connection.execute("SELECT * FROM patients ORDER BY name").fetchall()

    def get_patient(self, patient_id):
        return self.connection.execute("SELECT * FROM patients WHERE id=?", (patient_id,)).fetchone()

    # ---------------- Doctors ----------------
    def add_doctor(self, data):
        self.connection.execute("""
            INSERT INTO doctors(name, specialization, phone, room, email, availability)
            VALUES (?, ?, ?, ?, ?, ?)
        """, data)
        self.connection.commit()

    def update_doctor(self, doctor_id, data):
        self.connection.execute("""
            UPDATE doctors SET
                name=?, specialization=?, phone=?, room=?, email=?, availability=?
            WHERE id=?
        """, (*data, doctor_id))
        self.connection.commit()

    def delete_doctor(self, doctor_id):
        self.connection.execute("DELETE FROM doctors WHERE id=?", (doctor_id,))
        self.connection.commit()

    def get_doctors(self, search=""):
        if search:
            term = f"%{search}%"
            return self.connection.execute("""
                SELECT * FROM doctors
                WHERE name LIKE ? OR specialization LIKE ? OR phone LIKE ? OR CAST(id AS TEXT) LIKE ?
                ORDER BY name
            """, (term, term, term, term)).fetchall()
        return self.connection.execute("SELECT * FROM doctors ORDER BY name").fetchall()

    def get_doctor(self, doctor_id):
        return self.connection.execute("SELECT * FROM doctors WHERE id=?", (doctor_id,)).fetchone()

    # ---------------- Appointments ----------------
    def appointment_exists(self, doctor_id, appt_date, appt_time, exclude_id=None):
        sql = """
            SELECT id FROM appointments
            WHERE doctor_id=? AND appointment_date=? AND appointment_time=?
            AND status != 'Cancelled'
        """
        params = [doctor_id, appt_date, appt_time]
        if exclude_id is not None:
            sql += " AND id != ?"
            params.append(exclude_id)
        return self.connection.execute(sql, params).fetchone() is not None

    def add_appointment(self, data):
        patient_id, doctor_id, appt_date, appt_time, reason, notes = data
        if self.appointment_exists(doctor_id, appt_date, appt_time):
            raise ValueError("This doctor already has an appointment at this date and time.")
        self.connection.execute("""
            INSERT INTO appointments
            (patient_id, doctor_id, appointment_date, appointment_time, reason, status, notes)
            VALUES (?, ?, ?, ?, ?, 'Scheduled', ?)
        """, data)
        self.connection.commit()

    def update_appointment(self, appointment_id, data):
        patient_id, doctor_id, appt_date, appt_time, reason, notes = data
        if self.appointment_exists(doctor_id, appt_date, appt_time, appointment_id):
            raise ValueError("This doctor already has another appointment at this date and time.")
        self.connection.execute("""
            UPDATE appointments SET patient_id=?, doctor_id=?, appointment_date=?,
            appointment_time=?, reason=?, notes=? WHERE id=?
        """, (*data, appointment_id))
        self.connection.commit()

    def delete_appointment(self, appointment_id):
        self.connection.execute("DELETE FROM appointments WHERE id=?", (appointment_id,))
        self.connection.commit()

    def get_appointments(self, search="", appt_date=None):
        sql = """
            SELECT a.*, p.name AS patient_name, d.name AS doctor_name,
                   d.specialization, p.phone AS patient_phone
            FROM appointments a
            JOIN patients p ON a.patient_id=p.id
            JOIN doctors d ON a.doctor_id=d.id
            WHERE 1=1
        """
        params = []
        if appt_date:
            sql += " AND a.appointment_date=?"
            params.append(appt_date)
        if search:
            term = f"%{search}%"
            sql += " AND (p.name LIKE ? OR d.name LIKE ? OR a.reason LIKE ? OR a.status LIKE ? OR CAST(a.id AS TEXT) LIKE ?)"
            params.extend([term] * 5)
        sql += " ORDER BY a.appointment_date DESC, a.appointment_time ASC"
        return self.connection.execute(sql, params).fetchall()

    def get_appointment(self, appointment_id):
        return self.connection.execute("""
            SELECT a.*, p.name AS patient_name, p.age AS patient_age,
                   p.gender AS patient_gender, p.phone AS patient_phone,
                   p.address AS patient_address, p.email AS patient_email,
                   p.blood_group, p.allergies,
                   d.name AS doctor_name, d.specialization AS doctor_specialization
            FROM appointments a
            JOIN patients p ON a.patient_id=p.id
            JOIN doctors d ON a.doctor_id=d.id
            WHERE a.id=?
        """, (appointment_id,)).fetchone()

    def update_appointment_status(self, appointment_id, status):
        self.connection.execute("UPDATE appointments SET status=? WHERE id=?", (status, appointment_id))
        self.connection.commit()

    # ---------------- Consultations / prescriptions ----------------
    def get_consultation(self, appointment_id):
        return self.connection.execute("""
            SELECT * FROM consultations WHERE appointment_id=? ORDER BY id DESC LIMIT 1
        """, (appointment_id,)).fetchone()

    def save_consultation(self, appointment_id, data, medicines):
        self.connection.execute("DELETE FROM consultations WHERE appointment_id=?", (appointment_id,))
        cur = self.connection.execute("""
            INSERT INTO consultations
            (appointment_id, symptoms, diagnosis, prescription, doctor_notes,
             blood_pressure, temperature, weight, notes, follow_up_date)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, (appointment_id, *data))
        consultation_id = cur.lastrowid
        for medicine in medicines:
            if medicine[0].strip():
                self.connection.execute("""
                    INSERT INTO prescriptions
                    (consultation_id, medicine, dosage, frequency, duration, instructions)
                    VALUES (?, ?, ?, ?, ?, ?)
                """, (consultation_id, *medicine))
        self.connection.execute("UPDATE appointments SET status='Completed' WHERE id=?", (appointment_id,))
        self.connection.commit()

    def get_prescriptions(self, consultation_id):
        return self.connection.execute("""
            SELECT * FROM prescriptions WHERE consultation_id=? ORDER BY id
        """, (consultation_id,)).fetchall()

    def patient_history(self, patient_id):
        return self.connection.execute("""
            SELECT a.*, d.name AS doctor_name, d.specialization,
                   c.id AS consultation_id, c.symptoms, c.diagnosis,
                   c.prescription, c.doctor_notes, c.blood_pressure,
                   c.temperature, c.weight, c.notes AS consultation_notes,
                   c.follow_up_date
            FROM appointments a
            JOIN doctors d ON a.doctor_id=d.id
            LEFT JOIN consultations c ON c.appointment_id=a.id
            WHERE a.patient_id=?
            ORDER BY a.appointment_date DESC, a.appointment_time DESC
        """, (patient_id,)).fetchall()

    # ---------------- Dashboard / reports ----------------
    def count(self, table):
        return self.connection.execute(f"SELECT COUNT(*) AS n FROM {table}").fetchone()["n"]

    def count_today_status(self, status=None):
        today = date.today().strftime(DATE_FORMAT)
        if status:
            return self.connection.execute(
                "SELECT COUNT(*) AS n FROM appointments WHERE appointment_date=? AND status=?",
                (today, status)
            ).fetchone()["n"]
        return self.connection.execute(
            "SELECT COUNT(*) AS n FROM appointments WHERE appointment_date=?", (today,)
        ).fetchone()["n"]

    def upcoming_appointments(self, limit=8):
        return self.connection.execute("""
            SELECT a.*, p.name AS patient_name, d.name AS doctor_name
            FROM appointments a
            JOIN patients p ON a.patient_id=p.id
            JOIN doctors d ON a.doctor_id=d.id
            WHERE (a.appointment_date > ? OR (a.appointment_date=? AND a.appointment_time>=?))
              AND a.status NOT IN ('Cancelled', 'Completed')
            ORDER BY a.appointment_date, a.appointment_time
            LIMIT ?
        """, (date.today().strftime(DATE_FORMAT), date.today().strftime(DATE_FORMAT),
              datetime.now().strftime(TIME_FORMAT), limit)).fetchall()


In [4]:
class DoctorManagementApp:
    def __init__(self, root):
        self.root = root
        self.root.title(APP_TITLE)
        self.root.geometry("1280x780")
        self.root.minsize(1100, 680)
        self.db = Database()
        self.current_user = None
        self.setup_style()
        self.root.protocol("WM_DELETE_WINDOW", self.close_application)
        self.show_login()

    def setup_style(self):
        style = ttk.Style()
        try:
            style.theme_use("clam")
        except tk.TclError:
            pass
        style.configure("Treeview", rowheight=30, font=("Segoe UI", 10))
        style.configure("Treeview.Heading", font=("Segoe UI", 10, "bold"))
        style.configure("TButton", font=("Segoe UI", 10), padding=7)
        style.configure("Title.TLabel", font=("Segoe UI", 22, "bold"))
        style.configure("SubTitle.TLabel", font=("Segoe UI", 11))

    def clear(self):
        for widget in self.root.winfo_children():
            widget.destroy()

    # ---------------- Login ----------------
    def show_login(self):
        self.clear()
        frame = ttk.Frame(self.root, padding=30)
        frame.place(relx=.5, rely=.5, anchor="center")
        ttk.Label(frame, text="Doctor Management System", style="Title.TLabel").grid(row=0, column=0, columnspan=2, pady=(0, 8))
        ttk.Label(frame, text="Secure Login", style="SubTitle.TLabel").grid(row=1, column=0, columnspan=2, pady=(0, 25))
        ttk.Label(frame, text="Username").grid(row=2, column=0, sticky="w", pady=7)
        username = ttk.Entry(frame, width=32)
        username.grid(row=2, column=1, pady=7)
        ttk.Label(frame, text="Password").grid(row=3, column=0, sticky="w", pady=7)
        password = ttk.Entry(frame, width=32, show="*")
        password.grid(row=3, column=1, pady=7)
        def login():
            user = self.db.login(username.get().strip(), password.get())
            if not user:
                messagebox.showerror("Login Failed", "Invalid username or password.")
                return
            self.current_user = dict(user)
            self.show_main()
        ttk.Button(frame, text="LOGIN", command=login).grid(row=4, column=0, columnspan=2, sticky="ew", pady=20)
        ttk.Label(frame, text="Default login: admin / admin123").grid(row=5, column=0, columnspan=2)
        username.focus_set()
        password.bind("<Return>", lambda e: login())

    def logout(self):
        if messagebox.askyesno("Logout", "Do you want to logout?"):
            self.current_user = None
            self.show_login()

    def change_username_window(self):
        """Open a username-change form for the currently logged-in user."""
        if not self.current_user:
            return

        win = tk.Toplevel(self.root)
        win.title("Change Username")
        win.geometry("470x300")
        win.resizable(False, False)
        win.transient(self.root)
        win.grab_set()

        frame = ttk.Frame(win, padding=25)
        frame.pack(fill="both", expand=True)

        ttk.Label(frame, text="Change Username", style="Title.TLabel").grid(
            row=0, column=0, columnspan=2, pady=(0, 20)
        )
        ttk.Label(frame, text=f"Current Username: {self.current_user['username']}").grid(
            row=1, column=0, columnspan=2, sticky="w", pady=(0, 12)
        )

        ttk.Label(frame, text="New Username").grid(row=2, column=0, sticky="w", pady=8)
        username = ttk.Entry(frame, width=30)
        username.grid(row=2, column=1, pady=8)
        username.insert(0, self.current_user["username"])

        ttk.Label(frame, text="Current Password").grid(row=3, column=0, sticky="w", pady=8)
        password = ttk.Entry(frame, width=30, show="*")
        password.grid(row=3, column=1, pady=8)

        def save_username():
            new_username = username.get().strip()
            current_password = password.get()

            if not new_username or not current_password:
                messagebox.showerror(
                    "Validation", "Please enter the new username and current password.",
                    parent=win
                )
                return

            if new_username == self.current_user["username"]:
                messagebox.showerror(
                    "Validation", "The new username must be different from the current username.",
                    parent=win
                )
                return

            success, result = self.db.change_username(
                self.current_user["id"], current_password, new_username
            )

            if not success:
                messagebox.showerror("Change Username", result, parent=win)
                return

            # Keep the current session in sync with the database.
            self.current_user["username"] = result
            win.destroy()
            self.show_main()
            messagebox.showinfo(
                "Username Changed",
                f"Username changed successfully to: {result}"
            )

        ttk.Button(
            frame, text="Change Username", command=save_username
        ).grid(row=4, column=0, columnspan=2, sticky="ew", pady=(18, 5))
        ttk.Button(
            frame, text="Cancel", command=win.destroy
        ).grid(row=5, column=0, columnspan=2, sticky="ew")

        username.focus_set()
        password.bind("<Return>", lambda e: save_username())

    def change_password_window(self):
        """Open a secure password-change form for the currently logged-in user."""
        if not self.current_user:
            return

        win = tk.Toplevel(self.root)
        win.title("Change Password")
        win.geometry("470x330")
        win.resizable(False, False)
        win.transient(self.root)
        win.grab_set()

        frame = ttk.Frame(win, padding=25)
        frame.pack(fill="both", expand=True)

        ttk.Label(frame, text="Change Password", style="Title.TLabel").grid(
            row=0, column=0, columnspan=2, pady=(0, 20)
        )
        ttk.Label(frame, text=f"Username: {self.current_user['username']}").grid(
            row=1, column=0, columnspan=2, sticky="w", pady=(0, 15)
        )

        ttk.Label(frame, text="Current Password").grid(row=2, column=0, sticky="w", pady=8)
        current = ttk.Entry(frame, width=30, show="*")
        current.grid(row=2, column=1, pady=8)

        ttk.Label(frame, text="New Password").grid(row=3, column=0, sticky="w", pady=8)
        new = ttk.Entry(frame, width=30, show="*")
        new.grid(row=3, column=1, pady=8)

        ttk.Label(frame, text="Confirm Password").grid(row=4, column=0, sticky="w", pady=8)
        confirm = ttk.Entry(frame, width=30, show="*")
        confirm.grid(row=4, column=1, pady=8)

        def save_password():
            old_password = current.get()
            new_password = new.get()
            confirm_password = confirm.get()

            if not old_password or not new_password or not confirm_password:
                messagebox.showerror(
                    "Validation", "Please fill in all password fields.", parent=win
                )
                return

            if len(new_password) < 6:
                messagebox.showerror(
                    "Validation",
                    "New password must contain at least 6 characters.",
                    parent=win
                )
                return

            if new_password != confirm_password:
                messagebox.showerror(
                    "Validation", "New passwords do not match.", parent=win
                )
                return

            if old_password == new_password:
                messagebox.showerror(
                    "Validation",
                    "New password must be different from the current password.",
                    parent=win
                )
                return

            success = self.db.change_password(
                self.current_user["id"], old_password, new_password
            )

            if not success:
                messagebox.showerror(
                    "Change Password", "Current password is incorrect.", parent=win
                )
                current.focus_set()
                return

            messagebox.showinfo(
                "Password Changed",
                "Your password has been changed successfully.",
                parent=win
            )
            win.destroy()

        ttk.Button(frame, text="Change Password", command=save_password).grid(
            row=5, column=0, columnspan=2, sticky="ew", pady=(18, 5)
        )
        ttk.Button(frame, text="Cancel", command=win.destroy).grid(
            row=6, column=0, columnspan=2, sticky="ew"
        )

        current.focus_set()
        confirm.bind("<Return>", lambda e: save_password())

    # ---------------- Main layout ----------------
    def show_main(self):
        self.clear()
        top = ttk.Frame(self.root, padding=(15, 10))
        top.pack(fill="x")
        ttk.Label(top, text=APP_TITLE, style="Title.TLabel").pack(side="left")
        ttk.Label(top, text=f"Logged in: {self.current_user['username']} ({self.current_user['role']})").pack(side="right", padx=10)
        ttk.Button(top, text="Change Username", command=self.change_username_window).pack(side="right", padx=5)
        ttk.Button(top, text="Change Password", command=self.change_password_window).pack(side="right", padx=5)
        ttk.Button(top, text="Logout", command=self.logout).pack(side="right")

        nav = ttk.Frame(self.root, padding=(10, 0))
        nav.pack(fill="x")
        for text, command in [
            ("Dashboard", self.show_dashboard),
            ("Patients", self.show_patients),
            ("Doctors", self.show_doctors),
            ("Appointments", self.show_appointments),
            ("Consultations", self.show_consultations),
            ("Reports", self.show_reports),
        ]:
            ttk.Button(nav, text=text, command=command).pack(side="left", padx=3, pady=5)
        self.content = ttk.Frame(self.root, padding=15)
        self.content.pack(fill="both", expand=True)
        self.show_dashboard()

    def create_header(self, title, subtitle=""):
        ttk.Label(self.content, text=title, style="Title.TLabel").pack(anchor="w")
        if subtitle:
            ttk.Label(self.content, text=subtitle, style="SubTitle.TLabel").pack(anchor="w", pady=(0, 12))

    def clear_content(self):
        for widget in self.content.winfo_children():
            widget.destroy()

    def make_tree(self, parent, columns, headings, widths=None):
        frame = ttk.Frame(parent)
        frame.pack(fill="both", expand=True, pady=10)
        tree = ttk.Treeview(frame, columns=columns, show="headings")
        for i, col in enumerate(columns):
            tree.heading(col, text=headings[i])
            tree.column(col, width=(widths[i] if widths else 120), anchor="center")
        y = ttk.Scrollbar(frame, orient="vertical", command=tree.yview)
        x = ttk.Scrollbar(frame, orient="horizontal", command=tree.xview)
        tree.configure(yscrollcommand=y.set, xscrollcommand=x.set)
        tree.grid(row=0, column=0, sticky="nsew")
        y.grid(row=0, column=1, sticky="ns")
        x.grid(row=1, column=0, sticky="ew")
        frame.rowconfigure(0, weight=1)
        frame.columnconfigure(0, weight=1)
        return tree

    # ---------------- Dashboard ----------------
    def show_dashboard(self):
        self.clear_content()
        self.create_header("Dashboard", "Overview of your clinic")
        cards = ttk.Frame(self.content)
        cards.pack(fill="x", pady=5)
        values = [
            ("Patients", self.db.count("patients")),
            ("Doctors", self.db.count("doctors")),
            ("Today's Appointments", self.db.count_today_status()),
            ("Completed Today", self.db.count_today_status("Completed")),
            ("Scheduled Today", self.db.count_today_status("Scheduled")),
            ("Cancelled Today", self.db.count_today_status("Cancelled")),
        ]
        for i, (label, value) in enumerate(values):
            card = ttk.LabelFrame(cards, text=label, padding=15)
            card.grid(row=0, column=i, padx=5, sticky="nsew")
            ttk.Label(card, text=str(value), font=("Segoe UI", 22, "bold")).pack()
            cards.columnconfigure(i, weight=1)

        ttk.Label(self.content, text="Upcoming Appointments", font=("Segoe UI", 14, "bold")).pack(anchor="w", pady=(20, 5))
        tree = self.make_tree(self.content,
                              ("id", "date", "time", "patient", "doctor", "reason", "status"),
                              ("ID", "Date", "Time", "Patient", "Doctor", "Reason", "Status"),
                              (60, 100, 80, 180, 180, 230, 110))
        for row in self.db.upcoming_appointments():
            tree.insert("", "end", values=(row["id"], row["appointment_date"], row["appointment_time"],
                                             row["patient_name"], row["doctor_name"], row["reason"] or "", row["status"]))

    # ---------------- Patients ----------------
    def show_patients(self):
        self.clear_content()
        self.create_header("Patient Management", "Search, add, edit, delete, and view medical history")
        toolbar = ttk.Frame(self.content)
        toolbar.pack(fill="x")
        ttk.Label(toolbar, text="Search:").pack(side="left")
        search = ttk.Entry(toolbar, width=35)
        search.pack(side="left", padx=8)
        tree = self.make_tree(self.content,
                              ("id", "name", "age", "gender", "phone", "blood", "allergies", "email"),
                              ("ID", "Name", "Age", "Gender", "Phone", "Blood Group", "Allergies", "Email"),
                              (55, 190, 60, 80, 130, 100, 180, 200))
        def refresh(*_):
            for item in tree.get_children(): tree.delete(item)
            for p in self.db.get_patients(search.get().strip()):
                tree.insert("", "end", values=(p["id"], p["name"], p["age"] or "", p["gender"] or "",
                                                 p["phone"] or "", p["blood_group"] or "", p["allergies"] or "", p["email"] or ""))
        ttk.Button(toolbar, text="Add Patient", command=lambda: self.patient_form(refresh)).pack(side="left", padx=3)
        ttk.Button(toolbar, text="Edit", command=lambda: self.edit_selected_patient(tree, refresh)).pack(side="left", padx=3)
        ttk.Button(toolbar, text="Delete", command=lambda: self.delete_selected_patient(tree, refresh)).pack(side="left", padx=3)
        ttk.Button(toolbar, text="Medical History", command=lambda: self.patient_history_selected(tree)).pack(side="left", padx=3)
        search.bind("<KeyRelease>", refresh)
        refresh()

    def selected_id(self, tree):
        selected = tree.selection()
        if not selected:
            messagebox.showwarning("Selection", "Please select a record first.")
            return None
        return tree.item(selected[0])["values"][0]

    def patient_form(self, refresh, patient=None):
        win = tk.Toplevel(self.root)
        win.title("Edit Patient" if patient else "Add Patient")
        win.geometry("520x650")
        fields = [
            ("Name", "name"), ("Age", "age"), ("Gender", "gender"), ("Phone", "phone"),
            ("Email", "email"), ("Blood Group", "blood_group"), ("Emergency Contact", "emergency_contact"),
            ("Allergies", "allergies"), ("Address", "address")
        ]
        entries = {}
        form = ttk.Frame(win, padding=20); form.pack(fill="both", expand=True)
        for i, (label, key) in enumerate(fields):
            ttk.Label(form, text=label).grid(row=i, column=0, sticky="w", pady=6)
            if key in ("gender", "blood_group"):
                box = ttk.Combobox(form, width=35, state="readonly")
                box["values"] = (("Male", "Female", "Other") if key == "gender" else
                                  ("A+", "A-", "B+", "B-", "AB+", "AB-", "O+", "O-"))
                box.grid(row=i, column=1, sticky="ew", pady=6)
                entries[key] = box
            elif key in ("address", "allergies"):
                text = tk.Text(form, width=38, height=3)
                text.grid(row=i, column=1, sticky="ew", pady=6)
                entries[key] = text
            else:
                e = ttk.Entry(form, width=38); e.grid(row=i, column=1, sticky="ew", pady=6); entries[key] = e
        if patient:
            for key, widget in entries.items():
                value = patient[key] or ""
                if isinstance(widget, tk.Text): widget.insert("1.0", value)
                else: widget.set(value) if isinstance(widget, ttk.Combobox) else widget.insert(0, value)
        def value(key):
            w = entries[key]
            return w.get("1.0", "end").strip() if isinstance(w, tk.Text) else w.get().strip()
        def save():
            name = value("name")
            if not name:
                messagebox.showerror("Validation", "Patient name is required.", parent=win); return
            age = value("age")
            if age and (not age.isdigit() or int(age) < 0 or int(age) > 150):
                messagebox.showerror("Validation", "Enter a valid age.", parent=win); return
            data = (name, int(age) if age else None, value("gender"), value("phone"), value("address"),
                    value("email"), value("blood_group"), value("allergies"), value("emergency_contact"))
            if patient: self.db.update_patient(patient["id"], data)
            else: self.db.add_patient(data)
            win.destroy(); refresh()
        ttk.Button(form, text="Save Patient", command=save).grid(row=len(fields), column=0, columnspan=2, pady=20)

    def edit_selected_patient(self, tree, refresh):
        pid = self.selected_id(tree)
        if pid is not None: self.patient_form(refresh, self.db.get_patient(pid))

    def delete_selected_patient(self, tree, refresh):
        pid = self.selected_id(tree)
        if pid is not None and messagebox.askyesno("Delete", "Delete this patient and related appointments?"):
            self.db.delete_patient(pid); refresh()

    def patient_history_selected(self, tree):
        pid = self.selected_id(tree)
        if pid is None: return
        patient = self.db.get_patient(pid)
        history = self.db.patient_history(pid)
        win = tk.Toplevel(self.root); win.title(f"Medical History - {patient['name']}"); win.geometry("1050x650")
        ttk.Label(win, text=f"Medical History: {patient['name']}", style="Title.TLabel").pack(anchor="w", padx=15, pady=15)
        info = ttk.Label(win, text=f"Phone: {patient['phone'] or '-'}   |   Blood Group: {patient['blood_group'] or '-'}   |   Allergies: {patient['allergies'] or '-'}")
        info.pack(anchor="w", padx=15)
        tree2 = self.make_tree(win, ("date", "time", "doctor", "reason", "status", "diagnosis", "followup"),
                               ("Date", "Time", "Doctor", "Reason", "Status", "Diagnosis", "Follow-up"),
                               (100, 80, 170, 180, 100, 220, 110))
        for r in history:
            tree2.insert("", "end", values=(r["appointment_date"], r["appointment_time"], r["doctor_name"],
                                              r["reason"] or "", r["status"], r["diagnosis"] or "", r["follow_up_date"] or ""))

    # ---------------- Doctors ----------------
    def show_doctors(self):
        self.clear_content(); self.create_header("Doctor Management", "Manage doctors and availability")
        toolbar = ttk.Frame(self.content); toolbar.pack(fill="x")
        ttk.Label(toolbar, text="Search:").pack(side="left")
        search = ttk.Entry(toolbar, width=35); search.pack(side="left", padx=8)
        tree = self.make_tree(self.content, ("id", "name", "specialization", "phone", "room", "availability"),
                              ("ID", "Name", "Specialization", "Phone", "Room", "Availability"),
                              (55, 190, 190, 130, 90, 220))
        def refresh(*_):
            for item in tree.get_children(): tree.delete(item)
            for d in self.db.get_doctors(search.get().strip()):
                tree.insert("", "end", values=(d["id"], d["name"], d["specialization"] or "", d["phone"] or "", d["room"] or "", d["availability"] or ""))
        ttk.Button(toolbar, text="Add Doctor", command=lambda: self.doctor_form(refresh)).pack(side="left", padx=3)
        ttk.Button(toolbar, text="Edit", command=lambda: self.edit_selected_doctor(tree, refresh)).pack(side="left", padx=3)
        ttk.Button(toolbar, text="Delete", command=lambda: self.delete_selected_doctor(tree, refresh)).pack(side="left", padx=3)
        search.bind("<KeyRelease>", refresh); refresh()

    def doctor_form(self, refresh, doctor=None):
        win = tk.Toplevel(self.root); win.title("Edit Doctor" if doctor else "Add Doctor"); win.geometry("520x520")
        fields = [("Name", "name"), ("Specialization", "specialization"), ("Phone", "phone"), ("Email", "email"), ("Room", "room"), ("Availability", "availability")]
        entries = {}; form = ttk.Frame(win, padding=20); form.pack(fill="both", expand=True)
        for i, (label, key) in enumerate(fields):
            ttk.Label(form, text=label).grid(row=i, column=0, sticky="w", pady=7)
            e = ttk.Entry(form, width=40); e.grid(row=i, column=1, sticky="ew", pady=7); entries[key] = e
            if doctor: e.insert(0, doctor[key] or "")
        def save():
            data = tuple(entries[k].get().strip() for _, k in fields)
            if not data[0]: messagebox.showerror("Validation", "Doctor name is required.", parent=win); return
            if doctor: self.db.update_doctor(doctor["id"], data)
            else: self.db.add_doctor(data)
            win.destroy(); refresh()
        ttk.Button(form, text="Save Doctor", command=save).grid(row=len(fields), column=0, columnspan=2, pady=20)

    def edit_selected_doctor(self, tree, refresh):
        did = self.selected_id(tree)
        if did is not None: self.doctor_form(refresh, self.db.get_doctor(did))

    def delete_selected_doctor(self, tree, refresh):
        did = self.selected_id(tree)
        if did is not None and messagebox.askyesno("Delete", "Delete this doctor and related appointments?"):
            self.db.delete_doctor(did); refresh()

    # ---------------- Appointments ----------------
    def validate_date(self, value):
        try: datetime.strptime(value, DATE_FORMAT); return True
        except ValueError: return False

    def validate_time(self, value):
        try: datetime.strptime(value, TIME_FORMAT); return True
        except ValueError: return False

    def show_appointments(self):
        self.clear_content(); self.create_header("Appointment Management", "Schedule, reschedule, process, and cancel appointments")
        toolbar = ttk.Frame(self.content); toolbar.pack(fill="x")
        ttk.Label(toolbar, text="Search:").pack(side="left")
        search = ttk.Entry(toolbar, width=28); search.pack(side="left", padx=6)
        ttk.Label(toolbar, text="Date (YYYY-MM-DD):").pack(side="left", padx=(15, 5))
        date_filter = ttk.Entry(toolbar, width=14); date_filter.pack(side="left")
        tree = self.make_tree(self.content,
                              ("id", "date", "time", "patient", "doctor", "reason", "status"),
                              ("ID", "Date", "Time", "Patient", "Doctor", "Reason", "Status"),
                              (55, 100, 80, 190, 190, 230, 110))
        def refresh(*_):
            for item in tree.get_children(): tree.delete(item)
            for a in self.db.get_appointments(search.get().strip(), date_filter.get().strip() or None):
                tree.insert("", "end", values=(a["id"], a["appointment_date"], a["appointment_time"], a["patient_name"], a["doctor_name"], a["reason"] or "", a["status"]))
        ttk.Button(toolbar, text="Add", command=lambda: self.appointment_form(refresh)).pack(side="left", padx=3)
        ttk.Button(toolbar, text="Edit / Reschedule", command=lambda: self.appointment_form(refresh, self.get_selected_appointment(tree))).pack(side="left", padx=3)
        ttk.Button(toolbar, text="Process", command=lambda: self.process_appointment(tree, refresh)).pack(side="left", padx=3)
        ttk.Button(toolbar, text="Delete", command=lambda: self.delete_appointment(tree, refresh)).pack(side="left", padx=3)
        ttk.Button(toolbar, text="Consultation", command=lambda: self.open_consultation_from_tree(tree)).pack(side="left", padx=3)
        search.bind("<KeyRelease>", refresh); date_filter.bind("<KeyRelease>", refresh); refresh()

    def get_selected_appointment(self, tree):
        aid = self.selected_id(tree)
        return self.db.get_appointment(aid) if aid is not None else None

    def send_appointment_confirmation(self, appointment_id):
        """Send an appointment confirmation email to the patient's email address."""
        if not SMTP_USERNAME or not SMTP_PASSWORD:
            return False, "Email is not configured. Set SMTP_USERNAME and SMTP_PASSWORD."

        appt = self.db.get_appointment(appointment_id)
        if not appt:
            return False, "Appointment was not found."

        recipient = (appt["patient_email"] or "").strip()
        if not recipient:
            return False, "The patient does not have an email address."

        msg = EmailMessage()
        msg["Subject"] = f"Appointment Confirmation - {APP_TITLE}"
        msg["From"] = SMTP_USERNAME
        msg["To"] = recipient

        body = f"""Dear {appt['patient_name']},

Your appointment has been successfully scheduled.

Appointment Details
-------------------
Appointment ID : {appt['id']}
Doctor         : Dr. {appt['doctor_name']}
Specialization : {appt['doctor_specialization'] or 'N/A'}
Date           : {appt['appointment_date']}
Time           : {appt['appointment_time']}
Reason         : {appt['reason'] or 'N/A'}
Status         : Scheduled

Please arrive a few minutes before your appointment time.

Thank you,
{APP_TITLE}
"""
        msg.set_content(body)

        try:
            with smtplib.SMTP_SSL(SMTP_SERVER, SMTP_PORT, timeout=20) as server:
                server.login(SMTP_USERNAME, SMTP_PASSWORD)
                server.send_message(msg)
            return True, recipient
        except Exception as e:
            return False, str(e)

    def appointment_form(self, refresh, appt=None):
        if appt is False: return
        win = tk.Toplevel(self.root); win.title("Edit Appointment" if appt else "New Appointment"); win.geometry("560x560")
        form = ttk.Frame(win, padding=20); form.pack(fill="both", expand=True)
        patients = self.db.get_patients(); doctors = self.db.get_doctors()
        if not patients or not doctors:
            messagebox.showwarning("Missing Data", "Add at least one patient and one doctor first.", parent=win); win.destroy(); return
        ttk.Label(form, text="Patient").grid(row=0, column=0, sticky="w", pady=7)
        pbox = ttk.Combobox(form, width=42, state="readonly", values=[f"{p['id']} - {p['name']}" for p in patients]); pbox.grid(row=0, column=1, pady=7)
        ttk.Label(form, text="Doctor").grid(row=1, column=0, sticky="w", pady=7)
        dbox = ttk.Combobox(form, width=42, state="readonly", values=[f"{d['id']} - {d['name']} ({d['specialization'] or 'General'})" for d in doctors]); dbox.grid(row=1, column=1, pady=7)
        ttk.Label(form, text="Date (YYYY-MM-DD)").grid(row=2, column=0, sticky="w", pady=7)
        de = ttk.Entry(form); de.grid(row=2, column=1, pady=7); de.insert(0, appt["appointment_date"] if appt else date.today().strftime(DATE_FORMAT))
        ttk.Label(form, text="Time (HH:MM)").grid(row=3, column=0, sticky="w", pady=7)
        te = ttk.Entry(form); te.grid(row=3, column=1, pady=7); te.insert(0, appt["appointment_time"] if appt else "09:00")
        ttk.Label(form, text="Reason").grid(row=4, column=0, sticky="w", pady=7)
        reason = ttk.Entry(form, width=45); reason.grid(row=4, column=1, pady=7); reason.insert(0, appt["reason"] or "" if appt else "")
        ttk.Label(form, text="Notes").grid(row=5, column=0, sticky="nw", pady=7)
        notes = tk.Text(form, width=34, height=5); notes.grid(row=5, column=1, pady=7)
        if appt: notes.insert("1.0", appt["notes"] or "")
        if appt:
            for i, p in enumerate(patients):
                if p["id"] == appt["patient_id"]: pbox.current(i); break
            for i, d in enumerate(doctors):
                if d["id"] == appt["doctor_id"]: dbox.current(i); break
        def save():
            if pbox.current() < 0 or dbox.current() < 0:
                messagebox.showerror("Validation", "Select patient and doctor.", parent=win); return
            adate, atime = de.get().strip(), te.get().strip()
            if not self.validate_date(adate) or not self.validate_time(atime):
                messagebox.showerror("Validation", "Use date YYYY-MM-DD and time HH:MM.", parent=win); return
            pid = patients[pbox.current()]["id"]; did = doctors[dbox.current()]["id"]
            data = (pid, did, adate, atime, reason.get().strip(), notes.get("1.0", "end").strip())
            try:
                if appt:
                    self.db.update_appointment(appt["id"], data)
                    appointment_id = appt["id"]
                    email_sent = None
                    email_message = None
                else:
                    self.db.add_appointment(data)
                    appointment_id = self.db.connection.execute(
                        "SELECT last_insert_rowid()"
                    ).fetchone()[0]
                    email_sent, email_message = self.send_appointment_confirmation(appointment_id)
            except ValueError as e:
                messagebox.showerror("Appointment Conflict", str(e), parent=win); return

            win.destroy()
            refresh()

            # Confirmation is sent only for a newly scheduled appointment.
            if not appt:
                if email_sent:
                    messagebox.showinfo(
                        "Appointment Scheduled",
                        "Appointment scheduled successfully.\n\n"
                        f"Confirmation email sent to:\n{email_message}"
                    )
                else:
                    messagebox.showwarning(
                        "Appointment Scheduled",
                        "Appointment scheduled successfully, but the confirmation "
                        f"email could not be sent.\n\nReason: {email_message}"
                    )
        ttk.Button(form, text="Save Appointment", command=save).grid(row=6, column=0, columnspan=2, pady=20)

    def process_appointment(self, tree, refresh):
        aid = self.selected_id(tree)
        if aid is None: return
        appt = self.db.get_appointment(aid)
        win = tk.Toplevel(self.root); win.title("Process Appointment"); win.geometry("380x280")
        ttk.Label(win, text=f"{appt['patient_name']} with {appt['doctor_name']}", font=("Segoe UI", 13, "bold")).pack(pady=15)
        ttk.Label(win, text=f"{appt['appointment_date']} {appt['appointment_time']} | Current: {appt['status']}").pack()
        for status in ("Scheduled", "Completed", "Cancelled", "No Show"):
            ttk.Button(win, text=status, command=lambda s=status: self._set_status(aid, s, win, refresh)).pack(fill="x", padx=40, pady=4)

    def _set_status(self, aid, status, win, refresh):
        self.db.update_appointment_status(aid, status); win.destroy(); refresh()

    def delete_appointment(self, tree, refresh):
        aid = self.selected_id(tree)
        if aid is not None and messagebox.askyesno("Delete", "Delete this appointment?"):
            self.db.delete_appointment(aid); refresh()

    # ---------------- Consultations ----------------
    def show_consultations(self):
        self.clear_content(); self.create_header("Consultations", "Open a completed or scheduled appointment to record clinical information")
        tree = self.make_tree(self.content,
                              ("id", "date", "time", "patient", "doctor", "diagnosis", "status"),
                              ("ID", "Date", "Time", "Patient", "Doctor", "Diagnosis", "Status"),
                              (55, 100, 80, 190, 190, 250, 110))
        for a in self.db.get_appointments():
            c = self.db.get_consultation(a["id"])
            tree.insert("", "end", values=(a["id"], a["appointment_date"], a["appointment_time"], a["patient_name"],
                                             a["doctor_name"], c["diagnosis"] if c else "", a["status"]))
        ttk.Button(self.content, text="Open Consultation", command=lambda: self.open_consultation(tree)).pack(anchor="w")

    def open_consultation_from_tree(self, tree):
        aid = self.selected_id(tree)
        if aid is not None: self.consultation_window(aid)

    def open_consultation(self, tree):
        self.open_consultation_from_tree(tree)

    def send_prescription_email(self, appointment_id):
        """Email the completed consultation and prescription to the patient."""
        if not SMTP_USERNAME or not SMTP_PASSWORD:
            return False, "Email is not configured. Set SMTP_USERNAME and SMTP_PASSWORD."

        appt = self.db.get_appointment(appointment_id)
        if not appt:
            return False, "Appointment was not found."

        recipient = (appt["patient_email"] or "").strip()
        if not recipient:
            return False, "The patient does not have an email address."

        consultation = self.db.get_consultation(appointment_id)
        if not consultation:
            return False, "Consultation was not found."

        medicines = self.db.get_prescriptions(consultation["id"])

        msg = EmailMessage()
        msg["Subject"] = f"Prescription & Consultation - {APP_TITLE}"
        msg["From"] = SMTP_USERNAME
        msg["To"] = recipient

        medicine_lines = []
        for i, med in enumerate(medicines, start=1):
            medicine_lines.append(
                f"{i}. {med['medicine']} | Dosage: {med['dosage'] or 'N/A'} | "
                f"Frequency: {med['frequency'] or 'N/A'} | "
                f"Duration: {med['duration'] or 'N/A'} | "
                f"Instructions: {med['instructions'] or 'N/A'}"
            )
        medicine_text = "\n".join(medicine_lines) if medicine_lines else "No medicines were prescribed."

        body = f"""Dear {appt['patient_name']},

Your consultation with Dr. {appt['doctor_name']} has been completed.

Consultation Details
--------------------
Date           : {appt['appointment_date']}
Symptoms       : {consultation['symptoms'] or 'N/A'}
Diagnosis      : {consultation['diagnosis'] or 'N/A'}
Blood Pressure : {consultation['blood_pressure'] or 'N/A'}
Temperature    : {consultation['temperature'] or 'N/A'}
Weight         : {consultation['weight'] or 'N/A'}
Follow-up Date : {consultation['follow_up_date'] or 'N/A'}

Prescription / Medicines
------------------------
{medicine_text}

Doctor Notes
------------
{consultation['notes'] or consultation['doctor_notes'] or 'N/A'}

Please follow your doctor's instructions. Contact the clinic if you have any questions.

Thank you,
{APP_TITLE}
"""
        msg.set_content(body)

        # Also attach a PDF prescription so the patient receives a printable copy.
        try:
            if REPORTLAB_AVAILABLE:
                pdf_buffer = BytesIO()
                doc, styles = self.report_doc(pdf_buffer, "Prescription")
                story = [
                    Paragraph("Doctor Management System", styles["CenterTitle"]),
                    Paragraph("Prescription & Consultation", styles["Heading2"]),
                    Spacer(1, 10),
                    Paragraph(f"<b>Patient:</b> {appt['patient_name']}", styles["BodyText"]),
                    Paragraph(f"<b>Doctor:</b> {appt['doctor_name']}", styles["BodyText"]),
                    Paragraph(f"<b>Date:</b> {appt['appointment_date']}", styles["BodyText"]),
                    Spacer(1, 12),
                    Paragraph(f"<b>Symptoms:</b> {consultation['symptoms'] or 'N/A'}", styles["BodyText"]),
                    Paragraph(f"<b>Diagnosis:</b> {consultation['diagnosis'] or 'N/A'}", styles["BodyText"]),
                    Paragraph(f"<b>Follow-up:</b> {consultation['follow_up_date'] or 'N/A'}", styles["BodyText"]),
                    Spacer(1, 12),
                ]

                med_data = [["Medicine", "Dosage", "Frequency", "Duration", "Instructions"]]
                med_data += [
                    [
                        m["medicine"],
                        m["dosage"] or "",
                        m["frequency"] or "",
                        m["duration"] or "",
                        m["instructions"] or ""
                    ]
                    for m in medicines
                ]
                if len(med_data) == 1:
                    med_data.append(["No medicines prescribed", "", "", "", ""])

                table = Table(med_data, colWidths=[105, 75, 85, 75, 130])
                table.setStyle(TableStyle([
                    ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#d9eaf7")),
                    ("GRID", (0, 0), (-1, -1), .5, colors.grey),
                    ("VALIGN", (0, 0), (-1, -1), "TOP"),
                ]))
                story.append(table)
                story.append(Spacer(1, 12))
                story.append(
                    Paragraph(
                        f"<b>Doctor Notes:</b> {consultation['notes'] or consultation['doctor_notes'] or 'N/A'}",
                        styles["BodyText"]
                    )
                )
                doc.build(story)

                msg.add_attachment(
                    pdf_buffer.getvalue(),
                    maintype="application",
                    subtype="pdf",
                    filename=f"prescription_{appointment_id}.pdf"
                )

            with smtplib.SMTP_SSL(SMTP_SERVER, SMTP_PORT, timeout=20) as server:
                server.login(SMTP_USERNAME, SMTP_PASSWORD)
                server.send_message(msg)

            return True, recipient
        except Exception as e:
            return False, str(e)

    def consultation_window(self, appointment_id):
        appt = self.db.get_appointment(appointment_id)
        if not appt: messagebox.showerror("Error", "Appointment not found."); return
        existing = self.db.get_consultation(appointment_id)
        win = tk.Toplevel(self.root); win.title("Consultation"); win.geometry("800x720")
        ttk.Label(win, text="Consultation", style="Title.TLabel").pack(anchor="w", padx=20, pady=(15, 2))
        ttk.Label(win, text=f"Patient: {appt['patient_name']}    Doctor: {appt['doctor_name']}    Date: {appt['appointment_date']}").pack(anchor="w", padx=20, pady=(0, 12))
        form = ttk.Frame(win, padding=20); form.pack(fill="both", expand=True)
        fields = [("Symptoms", "symptoms"), ("Diagnosis", "diagnosis"), ("Blood Pressure", "blood_pressure"),
                  ("Temperature", "temperature"), ("Weight", "weight"), ("Follow-up Date", "follow_up_date"), ("Doctor Notes", "notes")]
        widgets = {}
        for i, (label, key) in enumerate(fields):
            ttk.Label(form, text=label).grid(row=i, column=0, sticky="nw", pady=6)
            if key in ("symptoms", "diagnosis", "notes"):
                w = tk.Text(form, width=55, height=4 if key != "notes" else 5); w.grid(row=i, column=1, pady=6, sticky="ew")
            else:
                w = ttk.Entry(form, width=55); w.grid(row=i, column=1, pady=6, sticky="ew")
            widgets[key] = w
            if existing:
                val = existing[key] or ""
                if isinstance(w, tk.Text): w.insert("1.0", val)
                else: w.insert(0, val)
        ttk.Label(form, text="Prescription / medicines").grid(row=len(fields), column=0, sticky="nw", pady=8)
        med_frame = ttk.Frame(form); med_frame.grid(row=len(fields), column=1, sticky="ew", pady=8)
        headers = ["Medicine", "Dosage", "Frequency", "Duration", "Instructions"]
        for j, h in enumerate(headers): ttk.Label(med_frame, text=h).grid(row=0, column=j, padx=2)
        med_entries = []
        existing_meds = self.db.get_prescriptions(existing["id"]) if existing else []
        rows = list(existing_meds) or [None]
        for r, med in enumerate(rows, start=1):
            row_widgets = []
            for j in range(5):
                e = ttk.Entry(med_frame, width=(18 if j == 0 else 14)); e.grid(row=r, column=j, padx=2, pady=2); row_widgets.append(e)
            if med:
                vals = [med["medicine"], med["dosage"], med["frequency"], med["duration"], med["instructions"]]
                for e, v in zip(row_widgets, vals): e.insert(0, v or "")
            med_entries.append(row_widgets)
        def add_med_row():
            r = len(med_entries) + 1; row_widgets=[]
            for j in range(5):
                e=ttk.Entry(med_frame,width=(18 if j==0 else 14)); e.grid(row=r,column=j,padx=2,pady=2); row_widgets.append(e)
            med_entries.append(row_widgets)
        ttk.Button(med_frame, text="+ Add Medicine", command=add_med_row).grid(row=100, column=0, columnspan=5, pady=6)
        def val(key):
            w=widgets[key]; return w.get("1.0","end").strip() if isinstance(w,tk.Text) else w.get().strip()
        def save():
            follow = val("follow_up_date")
            if follow and not self.validate_date(follow):
                messagebox.showerror("Validation", "Follow-up date must be YYYY-MM-DD.", parent=win); return
            data=(val("symptoms"), val("diagnosis"), existing["prescription"] if existing else "", val("notes"),
                  val("blood_pressure"), val("temperature"), val("weight"), val("notes"), follow)
            medicines=[tuple(e.get().strip() for e in row) for row in med_entries]
            self.db.save_consultation(appointment_id, data, medicines)

            # Automatically email the completed consultation and prescription.
            email_sent, email_result = self.send_prescription_email(appointment_id)

            win.destroy()
            if hasattr(self, "content"):
                self.show_consultations()

            if email_sent:
                messagebox.showinfo(
                    "Consultation Saved",
                    "Consultation saved and appointment marked Completed.\n\n"
                    f"Prescription email sent to:\n{email_result}"
                )
            else:
                messagebox.showwarning(
                    "Consultation Saved",
                    "Consultation saved and appointment marked Completed, "
                    "but the prescription email could not be sent.\n\n"
                    f"Reason: {email_result}"
                )
        ttk.Button(form, text="Save Consultation", command=save).grid(row=len(fields)+1, column=0, columnspan=2, pady=18)

    # ---------------- Reports ----------------
    def show_reports(self):
        self.clear_content(); self.create_header("Reports & Backup", "Generate PDFs, export appointments, and back up the database")
        frame = ttk.Frame(self.content); frame.pack(fill="x", pady=10)
        ttk.Button(frame, text="Patient History PDF", command=self.patient_report_prompt).pack(side="left", padx=5)
        ttk.Button(frame, text="Prescription PDF", command=self.prescription_report_prompt).pack(side="left", padx=5)
        ttk.Button(frame, text="Today's Appointments PDF", command=self.generate_today_report).pack(side="left", padx=5)
        ttk.Button(frame, text="Export Appointments CSV", command=self.export_appointments_csv).pack(side="left", padx=5)
        ttk.Button(frame, text="Backup Database", command=self.backup_database).pack(side="left", padx=5)
        ttk.Label(self.content, text="PDF reports require the ReportLab package.").pack(anchor="w", pady=10)

    def require_reportlab(self):
        if not REPORTLAB_AVAILABLE:
            messagebox.showerror("ReportLab Missing", "Install ReportLab with: pip install reportlab")
            return False
        return True

    def report_doc(self, path, title):
        doc = SimpleDocTemplate(path, pagesize=A4, rightMargin=35, leftMargin=35, topMargin=35, bottomMargin=35)
        styles = getSampleStyleSheet(); styles.add(ParagraphStyle(name="CenterTitle", parent=styles["Title"], alignment=TA_CENTER))
        return doc, styles

    def patient_report_prompt(self):
        patients=self.db.get_patients()
        if not patients: messagebox.showinfo("Report", "No patients available."); return
        win=tk.Toplevel(self.root); win.title("Patient Report"); win.geometry("430x160")
        ttk.Label(win,text="Select Patient").pack(pady=10)
        box=ttk.Combobox(win,state="readonly",width=42,values=[f"{p['id']} - {p['name']}" for p in patients]); box.pack()
        def go():
            if box.current()<0: return
            self.generate_patient_report_pdf(patients[box.current()]["id"]); win.destroy()
        ttk.Button(win,text="Generate PDF",command=go).pack(pady=15)

    def generate_patient_report_pdf(self, patient_id):
        if not self.require_reportlab(): return
        patient=self.db.get_patient(patient_id); history=self.db.patient_history(patient_id)
        path=filedialog.asksaveasfilename(defaultextension=".pdf",initialfile=f"patient_{patient_id}_history.pdf",filetypes=[("PDF","*.pdf")])
        if not path:return
        doc,styles=self.report_doc(path,"Patient Medical History")
        story=[Paragraph("Doctor Management System",styles["CenterTitle"]),Paragraph("Patient Medical History",styles["Heading2"]),Spacer(1,10)]
        story += [Paragraph(f"<b>Name:</b> {patient['name']}",styles["BodyText"]),Paragraph(f"<b>Age:</b> {patient['age'] or '-'} &nbsp; <b>Gender:</b> {patient['gender'] or '-'}",styles["BodyText"]),Paragraph(f"<b>Phone:</b> {patient['phone'] or '-'} &nbsp; <b>Blood Group:</b> {patient['blood_group'] or '-'}",styles["BodyText"]),Paragraph(f"<b>Allergies:</b> {patient['allergies'] or '-'}",styles["BodyText"]),Spacer(1,15)]
        data=[["Date","Doctor","Reason","Diagnosis","Follow-up"]]
        for r in history:data.append([r["appointment_date"],r["doctor_name"],r["reason"] or "",r["diagnosis"] or "",r["follow_up_date"] or ""])
        table=Table(data,colWidths=[65,100,120,150,75]);table.setStyle(TableStyle([("BACKGROUND",(0,0),(-1,0),colors.HexColor("#d9eaf7")),("GRID",(0,0),(-1,-1),.5,colors.grey),("VALIGN",(0,0),(-1,-1),"TOP")]));story.append(table);doc.build(story);messagebox.showinfo("Report",f"Saved to:\n{path}")

    def prescription_report_prompt(self):
        rows=[a for a in self.db.get_appointments() if self.db.get_consultation(a["id"])]
        if not rows: messagebox.showinfo("Report","No consultation prescriptions available."); return
        win=tk.Toplevel(self.root);win.title("Prescription Report");win.geometry("430x160")
        ttk.Label(win,text="Select consultation").pack(pady=10)
        box=ttk.Combobox(win,state="readonly",width=42,values=[f"{a['id']} - {a['patient_name']} - {a['appointment_date']}" for a in rows]);box.pack()
        def go():
            if box.current()<0:return
            self.generate_prescription_pdf(rows[box.current()]["id"]);win.destroy()
        ttk.Button(win,text="Generate PDF",command=go).pack(pady=15)

    def generate_prescription_pdf(self, appointment_id):
        if not self.require_reportlab(): return
        appt=self.db.get_appointment(appointment_id); c=self.db.get_consultation(appointment_id); meds=self.db.get_prescriptions(c["id"])
        path=filedialog.asksaveasfilename(defaultextension=".pdf",initialfile=f"prescription_{appointment_id}.pdf",filetypes=[("PDF","*.pdf")])
        if not path:return
        doc,styles=self.report_doc(path,"Prescription");story=[Paragraph("Doctor Management System",styles["CenterTitle"]),Paragraph("Prescription",styles["Heading2"]),Spacer(1,10)]
        story += [Paragraph(f"<b>Patient:</b> {appt['patient_name']}",styles["BodyText"]),Paragraph(f"<b>Doctor:</b> {appt['doctor_name']}",styles["BodyText"]),Paragraph(f"<b>Date:</b> {appt['appointment_date']}",styles["BodyText"]),Spacer(1,15)]
        data=[["Medicine","Dosage","Frequency","Duration","Instructions"]]+[[m["medicine"],m["dosage"] or "",m["frequency"] or "",m["duration"] or "",m["instructions"] or ""] for m in meds]
        table=Table(data,colWidths=[105,75,85,75,130]);table.setStyle(TableStyle([("BACKGROUND",(0,0),(-1,0),colors.HexColor("#d9eaf7")),("GRID",(0,0),(-1,-1),.5,colors.grey),("VALIGN",(0,0),(-1,-1),"TOP")]));story.append(table);doc.build(story);messagebox.showinfo("Report",f"Saved to:\n{path}")

    def generate_today_report(self):
        if not self.require_reportlab(): return
        today=date.today().strftime(DATE_FORMAT); rows=self.db.get_appointments(appt_date=today)
        path=filedialog.asksaveasfilename(defaultextension=".pdf",initialfile=f"appointments_{today}.pdf",filetypes=[("PDF","*.pdf")])
        if not path:return
        doc,styles=self.report_doc(path,"Today's Appointments");story=[Paragraph("Doctor Management System",styles["CenterTitle"]),Paragraph(f"Appointments for {today}",styles["Heading2"]),Spacer(1,12)]
        data=[["Time","Patient","Doctor","Reason","Status"]]+[[r["appointment_time"],r["patient_name"],r["doctor_name"],r["reason"] or "",r["status"]] for r in rows]
        table=Table(data,colWidths=[65,125,125,150,80]);table.setStyle(TableStyle([("BACKGROUND",(0,0),(-1,0),colors.HexColor("#d9eaf7")),("GRID",(0,0),(-1,-1),.5,colors.grey)]));story.append(table);doc.build(story);messagebox.showinfo("Report",f"Saved to:\n{path}")

    def export_appointments_csv(self):
        path=filedialog.asksaveasfilename(defaultextension=".csv",initialfile="appointments.csv",filetypes=[("CSV","*.csv")])
        if not path:return
        rows=self.db.get_appointments()
        with open(path,"w",newline="",encoding="utf-8") as f:
            writer=csv.writer(f);writer.writerow(["ID","Date","Time","Patient","Doctor","Reason","Status","Notes"])
            for r in rows: writer.writerow([r["id"],r["appointment_date"],r["appointment_time"],r["patient_name"],r["doctor_name"],r["reason"] or "",r["status"],r["notes"] or ""])
        messagebox.showinfo("Export",f"CSV saved to:\n{path}")

    def backup_database(self):
        path=filedialog.asksaveasfilename(defaultextension=".db",initialfile=f"doctor_management_backup_{datetime.now().strftime('%Y%m%d_%H%M%S')}.db",filetypes=[("SQLite Database","*.db")])
        if not path:return
        self.db.connection.commit();shutil.copy2(self.db.db_file,path);messagebox.showinfo("Backup",f"Database backup saved to:\n{path}")

    def close_application(self):
        try:self.db.close()
        finally:self.root.destroy()


In [ ]:
def main():
    root=tk.Tk()
    DoctorManagementApp(root)
    root.mainloop()

if __name__ == "__main__":
    main()
